# UAV Traffic Vision — YOLO26s on VisDrone2019-DET (Colab training)

One-click training notebook: **Runtime → Run all**.

What it does:
1. Trains `yolo26s` at **imgsz=640** on VisDrone2019-DET (auto-downloaded, ~2.3 GB, 6,471 train / 548 val images, 10 classes).
2. Checkpoints land directly on **Google Drive** (`MyDrive/uav-traffic-vision/runs/...`), so a disconnect never loses progress — see the resume note below.
3. Validates `best.pt` and copies it plus `results.csv` to a stable Drive path.
4. **Optional imgsz=1024 comparison** at the end: a short probe measures the real epoch time first and prints the projected hours / compute-unit cost, and the full run only starts after you flip `RUN_1024 = True`.

Requirements: Colab Pro, GPU runtime (**L4** recommended — Runtime → Change runtime type).

In [ ]:
# GPU / CPU check
import os
import subprocess
import torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available(), "No GPU — Runtime -> Change runtime type -> GPU"
GPU_NAME = torch.cuda.get_device_name(0)
print(f"CPU cores: {os.cpu_count()}")

In [ ]:
# Mount Google Drive (checkpoints and final weights live here)
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Configuration
from pathlib import Path

MODEL    = "yolo26s.pt"
IMGSZ    = 640
EPOCHS   = 100
PATIENCE = 20

# batch tuned per GPU: L4 has ~300 GB/s memory bandwidth and measured SLOWER
# above batch=16 (~50 img/s @16 vs ~35 img/s via AutoBatch's ~32 -- a bandwidth/
# cuDNN-algorithm wall, not an under-utilized GPU). A100 (80GB on Colab) has
# ~5-6x the bandwidth and much more VRAM; batch=64 measured ~100 img/s there
# (2x the L4 throughput), with GPU_mem settling around 33G/80G -- plenty of
# headroom left if you want to push higher.
if "A100" in GPU_NAME:
    BATCH = 64
elif "L4" in GPU_NAME:
    BATCH = 16
else:
    BATCH = 16  # conservative default for an unrecognized GPU; check GPU_mem in
                # the first epoch log and raise if there's clear headroom

CACHE    = "disk"  # cache decoded images on local disk: cuts repeat JPEG-decode
                    # + Drive/network read cost on every epoch after the first
                    # (measured: val step dropped from 9.1s to 2.4s after warmup)
SEED     = 42
RESUME   = False   # after a disconnect: set True, Runtime -> Run all again

DRIVE_ROOT  = Path("/content/drive/MyDrive/uav-traffic-vision")
RUNS_DIR    = DRIVE_ROOT / "runs"     # ultralytics project dir -> checkpoints on Drive
WEIGHTS_DIR = DRIVE_ROOT / "weights"  # stable copies of best.pt / results.csv
RUN_NAME    = f"visdrone_yolo26s_{IMGSZ}"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"GPU={GPU_NAME}  BATCH={BATCH}  cache={CACHE}")

In [ ]:
# Install ultralytics (pinned to the version locked in the repo's uv.lock)
%pip install -q ultralytics==8.4.90

from ultralytics import settings
settings.update({"datasets_dir": "/content/datasets"})  # Colab local disk, NOT Drive (fast I/O)

In [ ]:
# Download + convert VisDrone2019-DET (~2.3 GB on first run, built-in ultralytics converter)
from ultralytics.data.utils import check_det_dataset

data = check_det_dataset("VisDrone.yaml", autodownload=True)
print("train:", data["train"])
print("val:  ", data["val"])

## Train (imgsz=640 baseline)

Checkpoints are written to Drive every epoch. **If Colab disconnects**: reconnect,
set `RESUME = True` in the configuration cell, then Runtime → Run all — training
continues from `last.pt`.

In [ ]:
import time
from ultralytics import YOLO

last_pt = RUNS_DIR / RUN_NAME / "weights" / "last.pt"
if RESUME and last_pt.exists():
    print(f"resuming from {last_pt}")
    model = YOLO(str(last_pt))
    model.train(resume=True)
else:
    model = YOLO(MODEL)
    t0 = time.time()
    model.train(
        data="VisDrone.yaml", imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE,
        batch=BATCH, cache=CACHE, seed=SEED, project=str(RUNS_DIR), name=RUN_NAME,
        exist_ok=True,
    )
    print(f"training wall time: {(time.time() - t0) / 3600:.2f} h")

In [ ]:
# Validate the best checkpoint
best_pt = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
metrics = YOLO(str(best_pt)).val(data="VisDrone.yaml", imgsz=IMGSZ, split="val")
print(f"val mAP50={metrics.box.map50:.4f}  mAP50-95={metrics.box.map:.4f}")

In [ ]:
# Copy artifacts to a stable Drive path
import shutil

shutil.copy2(best_pt, WEIGHTS_DIR / f"best_{IMGSZ}.pt")
results_csv = RUNS_DIR / RUN_NAME / "results.csv"
if results_csv.exists():
    shutil.copy2(results_csv, WEIGHTS_DIR / f"results_{RUN_NAME}.csv")
print("artifacts on Drive:")
for p in sorted(WEIGHTS_DIR.iterdir()):
    print(" ", p)
print(f"\nNext: download best_{IMGSZ}.pt into the repo's weights/ folder (as best.pt) for Phase 2.")

## Optional: imgsz=1024 comparison experiment

VisDrone images are ~1400–2000 px wide; at imgsz=640 a distant pedestrian shrinks to a
few pixels. This experiment trains the same model at **imgsz=1024** for a resolution
ablation in the README.

Two-step gate, so nothing expensive runs by accident:
1. The **estimate cell** below (runs as part of Run-all, ~10 min): trains 2 probe epochs
   at 1024 to measure the real epoch time, then prints the projected total hours and
   compute-unit cost. Check the actual consumption rate in Colab under
   **View resources → compute units per hour** and adjust `L4_CU_PER_HOUR` if it differs.
2. Review the printout, then set `RUN_1024 = True` in the last cell and run it to start
   the full training. The 1024 weights are saved separately (`best_1024.pt`), never
   overwriting the 640 run.

In [ ]:
# Estimate imgsz=1024 cost (2-epoch probe, throwaway run on local disk)
ESTIMATE_1024 = True   # set False to skip the ~5 min probe

# batch and compute-unit rate depend on the attached GPU, same logic as BATCH
# above. L4 numbers are untried at 1024 (kept conservative); A100 80GB had
# 13.3G/80G at batch=8 for imgsz=1024, so 32 has ample headroom.
if "A100" in GPU_NAME:
    BATCH_HI, CU_PER_HOUR = 32, 15.0  # verify actual rate: View resources in Colab
elif "L4" in GPU_NAME:
    BATCH_HI, CU_PER_HOUR = 8, 2.4
else:
    BATCH_HI, CU_PER_HOUR = 8, 2.4

IMGSZ_HI, EPOCHS_HI = 1024, 100

if ESTIMATE_1024:
    import time
    import pandas as pd

    probe_dir = Path("/content/probe_runs")
    t0 = time.time()
    YOLO(MODEL).train(
        data="VisDrone.yaml", imgsz=IMGSZ_HI, epochs=2, batch=BATCH_HI, seed=SEED,
        project=str(probe_dir), name="probe_1024", exist_ok=True,
        val=False, plots=False, save=False,
    )
    wall = time.time() - t0
    try:  # per-epoch time from results.csv (2nd epoch, excludes warmup)
        t = pd.read_csv(probe_dir / "probe_1024" / "results.csv")["time"]
        epoch_s = float(t.iloc[1] - t.iloc[0])
    except Exception:
        epoch_s = wall / 2
    total_h = epoch_s * EPOCHS_HI / 3600
    print(f"\nmeasured epoch time @1024 (batch={BATCH_HI}) on {GPU_NAME}: {epoch_s / 60:.1f} min")
    print(f"projected full run, {EPOCHS_HI} epochs (no early stop): ~{total_h:.1f} h "
          f"~= {total_h * CU_PER_HOUR:.0f} compute units @ {CU_PER_HOUR}/h")
    print(f"patience={PATIENCE} usually stops earlier, so treat this as an upper bound.")
    print("If acceptable: set RUN_1024 = True in the next cell and run it.")
else:
    print("ESTIMATE_1024 = False — probe skipped")

In [ ]:
# Full imgsz=1024 training — only runs after you flip the switch
RUN_1024 = False  # review the estimate above, then set True and run this cell

# A100 80GB probe at BATCH_HI=32 hit GPU_mem 74.7G/80G (93%) and was still
# climbing epoch-to-epoch -- real OOM risk over a multi-hour run. Back off to
# 24 for headroom; only applies when the probe picked BATCH_HI=32 (A100).
if BATCH_HI == 32:
    BATCH_HI = 24

if RUN_1024:
    import shutil

    run_name_hi = f"visdrone_yolo26s_{IMGSZ_HI}"
    last_hi = RUNS_DIR / run_name_hi / "weights" / "last.pt"
    if RESUME and last_hi.exists():
        print(f"resuming from {last_hi}")
        YOLO(str(last_hi)).train(resume=True)
    else:
        YOLO(MODEL).train(
            data="VisDrone.yaml", imgsz=IMGSZ_HI, epochs=EPOCHS_HI, patience=PATIENCE,
            batch=BATCH_HI, cache=CACHE, seed=SEED, project=str(RUNS_DIR), name=run_name_hi,
            exist_ok=True,
        )
    best_hi = RUNS_DIR / run_name_hi / "weights" / "best.pt"
    m = YOLO(str(best_hi)).val(data="VisDrone.yaml", imgsz=IMGSZ_HI, split="val")
    print(f"1024: val mAP50={m.box.map50:.4f}  mAP50-95={m.box.map:.4f}")
    shutil.copy2(best_hi, WEIGHTS_DIR / f"best_{IMGSZ_HI}.pt")
    rc_hi = RUNS_DIR / run_name_hi / "results.csv"
    if rc_hi.exists():
        shutil.copy2(rc_hi, WEIGHTS_DIR / f"results_{run_name_hi}.csv")
    print("saved best_1024.pt to Drive")
else:
    print("RUN_1024 = False — skipped (review the estimate above first)")